# 01. INGESTAO DOS DADOS DO RENAEST

ESTE NOTEBOOK BAIXA OS DADOS ABERTOS DE ACIDENTES DO RENAEST E CARREGA NO BANCO SQLITE DO PROJETO.

E O PRIMEIRO DA SEQUENCIA. NADA DEPENDE DELE PARA EXISTIR, MAS O NOTEBOOK 03 DEPENDE DA TABELA QUE ELE CRIA.

A IDEIA CENTRAL E SIMPLES:

1. PERGUNTAR AO PORTAL DE DADOS ABERTOS QUAL E O ARQUIVO MAIS RECENTE.
2. BAIXAR O ARQUIVO ZIP DESSE RECURSO.
3. EXTRAIR O ZIP EM UMA PASTA TEMPORARIA.
4. CONFERIR QUAIS COLUNAS VIERAM DE VERDADE NO CSV.
5. CARREGAR OS CSV NO BANCO SQLITE, UMA TABELA POR ARQUIVO.
6. MOSTRAR UM RESUMO DO QUE FOI GRAVADO.

O PASSO 4 EXISTE POR UM MOTIVO ESPECIFICO: OS NOTEBOOKS 03 E 04 ASSUMEM NOMES DE COLUNA COMO `end_acidente` E `codigo_ibge`, MAS ESSES NOMES NUNCA FORAM CONFERIDOS CONTRA O ARQUIVO REAL. E MELHOR DESCOBRIR AQUI, COM UMA MENSAGEM CLARA, DO QUE NO MEIO DO NOTEBOOK 03.

## 1. IMPORTACAO DAS BIBLIOTECAS E CONFIGURACOES

AQUI CARREGAMOS AS BIBLIOTECAS E DEFINIMOS OS VALORES QUE CONTROLAM ESTE NOTEBOOK.

EXPLICACAO DOS VALORES MAIS IMPORTANTES:

- `URL_PACOTE`: ENDERECO DA API DO PORTAL DE DADOS ABERTOS QUE LISTA OS ARQUIVOS DISPONIVEIS DO RENAEST.
- `TAMANHO_CHUNK`: QUANTAS LINHAS O PANDAS LE POR VEZ. O CSV DO RENAEST E GRANDE E LER TUDO DE UMA VEZ PODE ESTOURAR A MEMORIA.
- `PREFIXOS_ARQUIVOS_PROCESSADOS`: QUAIS ARQUIVOS DO ZIP SERAO CARREGADOS. HOJE SO OS QUE COMECAM COM `Acidentes`. PARA CARREGAR TUDO, COLOQUE `None`.
- `USAR_ARQUIVOS_LOCAIS_SE_EXISTIREM`: SE O ZIP JA FOI BAIXADO ANTES, NAO BAIXA DE NOVO. O DOWNLOAD E A PARTE DEMORADA.

TODOS ESSES VALORES VEM DO ARQUIVO `parametros.py`, E NENHUM DELES E ESCRITO AQUI. OS CAMINHOS E O BANCO VEM DO `config.py`. A DIVISAO E: `parametros.py` E O QUE VOCE QUER PROCESSAR, `config.py` E ONDE AS COISAS ESTAO.

In [1]:
# RE E A BIBLIOTECA DE EXPRESSOES REGULARES, USADA PARA LIMPAR NOMES DE ARQUIVO.
import re

# SQLITE3 E O BANCO DE DADOS LOCAL, QUE VEM JUNTO COM O PYTHON.
import sqlite3

# SYS E USADO PARA GARANTIR QUE O PYTHON ENCONTRE O ARQUIVO config.py.
import sys

# ZIPFILE ABRE E EXTRAI ARQUIVOS .ZIP.
import zipfile

# PATH AJUDA A TRABALHAR COM CAMINHOS DE ARQUIVO DE FORMA MAIS SEGURA.
from pathlib import Path

# PANDAS E USADO PARA LER OS CSV E GRAVAR NO BANCO.
import pandas as pd

# REQUESTS FAZ AS CHAMADAS HTTP: CONSULTAR A API E BAIXAR O ZIP.
import requests

# GARANTE QUE A PASTA projeto_v2 ESTEJA NO CAMINHO DE IMPORTACAO,
# INDEPENDENTE DE ONDE O JUPYTER FOI ABERTO.
for _candidata in (Path.cwd(), Path.cwd() / "projeto_v2", Path.cwd().parent):
    if (_candidata / "config.py").exists():
        sys.path.insert(0, str(_candidata))
        break

# CONFIGURACAO COMPARTILHADA: CAMINHOS, BANCO, CHAVES E NOMES DE COLUNA.
import config

# O QUE SERA PROCESSADO: MUNICIPIO, ANO E AS DEMAIS ESCOLHAS. TODAS ELAS VIVEM NO
# parametros.py -- E O UNICO ARQUIVO QUE VOCE EDITA ANTES DE RODAR.
import parametros

# =========================
# PARAMETROS, TODOS VINDOS DO parametros.py
# =========================

URL_PACOTE = parametros.URL_PACOTE
TAMANHO_CHUNK = parametros.TAMANHO_CHUNK
PREFIXOS_ARQUIVOS_PROCESSADOS = parametros.PREFIXOS_ARQUIVOS_PROCESSADOS
USAR_ARQUIVOS_LOCAIS_SE_EXISTIREM = parametros.USAR_ARQUIVOS_LOCAIS_SE_EXISTIREM

# MOSTRA A CONFIGURACAO ATIVA PARA CONFERENCIA.
config.resumo()
print()
parametros.resumo()

KeyboardInterrupt: 

## 2. FUNCOES AUXILIARES

AQUI CRIAMOS AS FUNCOES QUE SERAO USADAS NAS ETAPAS SEGUINTES. NENHUMA DELAS EXECUTA NADA SOZINHA.

DESTAQUE PARA DUAS DECISOES QUE NAO SAO OBVIAS:

### COMO DESCOBRIMOS O ARQUIVO MAIS RECENTE

O PORTAL DEVOLVE UMA LISTA DE RECURSOS. A VERSAO ANTIGA DESTE CODIGO PEGAVA SIMPLESMENTE O ULTIMO ITEM DA LISTA, ASSUMINDO QUE A ORDEM ERA CRONOLOGICA.

ISSO E FRAGIL: SE O PORTAL REORDENAR A LISTA, O SCRIPT BAIXA O ARQUIVO ERRADO E NAO DA ERRO NENHUM. AQUI ORDENAMOS PELA DATA (`last_modified` OU `created`) E SO CAIMOS NO ULTIMO DA LISTA SE NENHUM RECURSO TIVER DATA.

### COMO O NOME DA TABELA E DECIDIDO

O ARQUIVO VEM COM A DATA NO NOME, POR EXEMPLO `Acidentes_DadosAbertos_20260612.csv`.

SE USASSEMOS O NOME INTEIRO, CADA MES CRIARIA UMA TABELA NOVA E NADA CONTINUARIA FUNCIONANDO. ENTAO CORTAMOS TUDO A PARTIR DE `_DadosAbertos` E FICAMOS COM A TABELA `acidentes`, QUE E ESTAVEL ENTRE AS ATUALIZACOES.

In [ ]:
def reportar(etapa=None, progresso=None, mensagem=None):
    """IMPRIME O ANDAMENTO DE UMA ETAPA.

    NA VERSAO EM API ESTA FUNCAO ALIMENTAVA UMA BARRA DE PROGRESSO REMOTA.
    NO NOTEBOOK ELA APENAS IMPRIME, MAS A ASSINATURA FOI MANTIDA IGUAL PARA
    QUE O CORPO DAS DEMAIS FUNCOES NAO PRECISASSE MUDAR.
    """
    partes = []
    if etapa:
        partes.append(f"[{etapa}]")
    if progresso is not None:
        partes.append(f"{progresso:5.1f}%")
    if mensagem:
        partes.append(mensagem)
    print(" ".join(partes))


def descobrir_recurso_mais_recente():
    """CONSULTA O PORTAL E DEVOLVE O RECURSO ATIVO MAIS RECENTE (URL + NOME)."""
    # PERGUNTA AO PORTAL QUAIS ARQUIVOS EXISTEM NO PACOTE DO RENAEST.
    resposta = requests.get(URL_PACOTE, timeout=30)

    # LEVANTA ERRO SE A RESPOSTA NAO FOR DE SUCESSO.
    resposta.raise_for_status()

    # CONVERTE A RESPOSTA JSON EM DICIONARIO PYTHON.
    dados = resposta.json()

    # MANTEM APENAS OS RECURSOS MARCADOS COMO ATIVOS.
    recursos_ativos = [r for r in dados["result"]["resources"] if r.get("state") == "active"]

    # SEM RECURSO ATIVO NAO HA O QUE BAIXAR.
    if not recursos_ativos:
        raise RuntimeError("NENHUM RECURSO ATIVO ENCONTRADO NO PACOTE RENAEST.")

    # ORDENA PELA DATA DE MODIFICACAO OU CRIACAO. RECURSO SEM DATA VAI PARA O INICIO.
    def _data(recurso):
        return recurso.get("last_modified") or recurso.get("created") or ""

    com_data = [r for r in recursos_ativos if _data(r)]

    if com_data:
        # O MAIS RECENTE E O ULTIMO DEPOIS DE ORDENAR PELA DATA.
        mais_recente = sorted(com_data, key=_data)[-1]
    else:
        # NENHUM RECURSO TEM DATA: SO ENTAO CONFIAMOS NA ORDEM DA LISTA.
        print("AVISO: NENHUM RECURSO TEM DATA. USANDO O ULTIMO DA LISTA.")
        mais_recente = recursos_ativos[-1]

    # EXTRAI A URL E MONTA O NOME LIMPO, SEM A EXTENSAO .zip.
    url = mais_recente["url"]
    arquivo = url.split("/")[-1]
    nome_limpo = arquivo.replace(".zip", "")

    return {
        "url": url,
        "arquivo": arquivo,
        "nome": nome_limpo,
        "data": _data(mais_recente) or "SEM DATA",
        "total_ativos": len(recursos_ativos),
    }


def baixar_arquivo(url, destino):
    """BAIXA UM ARQUIVO EM PEDACOS, MOSTRANDO O PERCENTUAL NA MESMA LINHA."""
    destino = Path(destino)

    # GARANTE QUE A PASTA DE DESTINO EXISTA.
    destino.parent.mkdir(parents=True, exist_ok=True)

    # stream=True FAZ O DOWNLOAD EM PEDACOS, SEM CARREGAR TUDO NA MEMORIA.
    with requests.get(url, stream=True, timeout=60) as resposta:
        resposta.raise_for_status()

        # TAMANHO TOTAL INFORMADO PELO SERVIDOR. PODE VIR ZERO.
        total = int(resposta.headers.get("content-length", 0))
        baixado = 0

        # PEDACOS DE 10 MB.
        tamanho_bloco = 1024 * 1024 * 10

        with open(destino, "wb") as arquivo_local:
            for bloco in resposta.iter_content(chunk_size=tamanho_bloco):
                # ALGUNS BLOCOS PODEM VIR VAZIOS, ENTAO IGNORAMOS.
                if not bloco:
                    continue

                # GRAVA O PEDACO NO ARQUIVO E SOMA O QUE JA FOI BAIXADO.
                arquivo_local.write(bloco)
                baixado += len(bloco)

                # IMPRIME O PROGRESSO NA MESMA LINHA, SO SE O TOTAL FOR CONHECIDO.
                if total > 0:
                    print(f"\rPROGRESSO: {baixado / total * 100:.2f}%", end="")

    print("\nDOWNLOAD CONCLUIDO.")
    return destino


def extrair_zip(caminho_zip, pasta_destino):
    """EXTRAI O ZIP BAIXADO PARA A PASTA DE DESTINO."""
    pasta_destino = Path(pasta_destino)

    # CRIA A PASTA DE DESTINO SE ELA AINDA NAO EXISTIR.
    pasta_destino.mkdir(parents=True, exist_ok=True)

    # ABRE O ARQUIVO .ZIP E EXTRAI TODO O CONTEUDO PARA A PASTA.
    with zipfile.ZipFile(caminho_zip, "r") as zip_ref:
        zip_ref.extractall(pasta_destino)

    print(f"EXTRACAO CONCLUIDA EM: {pasta_destino}")
    return pasta_destino


def sanitizar_nome_tabela(nome):
    """TRANSFORMA UM TEXTO QUALQUER EM UM NOME DE TABELA VALIDO PARA SQLITE."""
    # TROCA QUALQUER COISA QUE NAO SEJA LETRA OU NUMERO POR UNDERLINE.
    tabela = re.sub(r"[^0-9a-zA-Z]+", "_", nome.strip().lower()).strip("_")

    # SE SOBROU VAZIO, USA UM NOME GENERICO.
    if not tabela:
        tabela = "tabela"

    # O SQLITE NAO ACEITA NOME DE TABELA COMECANDO COM NUMERO.
    if tabela[0].isdigit():
        tabela = "t_" + tabela

    return tabela


def nome_tabela_limpo(stem):
    """REMOVE O SUFIXO '_DadosAbertos_<data>' PARA UM NOME ESTAVEL ENTRE MESES.

    EXEMPLO: 'Acidentes_DadosAbertos_20260612' VIRA A TABELA 'acidentes'.
    """
    # CORTA O NOME NO TRECHO '_dadosabertos', IGNORANDO MAIUSCULA E MINUSCULA.
    base = re.split(r"_dadosabertos", stem, flags=re.IGNORECASE)[0]

    # SE O CORTE DEIXOU VAZIO, USA O NOME ORIGINAL.
    return sanitizar_nome_tabela(base or stem)


def detectar_encoding_e_separador(caminho):
    """DESCOBRE O ENCODING E O SEPARADOR OLHANDO SOMENTE A PRIMEIRA LINHA."""
    encoding = "utf-8"

    # PRIMEIRO TENTA LER COMO UTF-8, QUE E O MAIS COMUM.
    try:
        with open(caminho, "r", encoding="utf-8") as arquivo:
            primeira_linha = arquivo.readline()
    except UnicodeDecodeError:
        # ARQUIVOS DO GOVERNO COSTUMAM VIR EM LATIN-1 (TAMBEM CHAMADO ISO-8859-1).
        encoding = "latin-1"
        with open(caminho, "r", encoding="latin-1") as arquivo:
            primeira_linha = arquivo.readline()

    # CONTA QUANTAS VEZES CADA SEPARADOR CANDIDATO APARECE NO CABECALHO.
    candidatos = {sep: primeira_linha.count(sep) for sep in (";", ",", "\t", "|")}

    # O SEPARADOR MAIS FREQUENTE E, QUASE SEMPRE, O CORRETO.
    separador = max(candidatos, key=candidatos.get)

    # SE NENHUM APARECEU, USA O PONTO E VIRGULA, PADRAO EM DADO BRASILEIRO.
    if candidatos[separador] == 0:
        separador = ";"

    return encoding, separador


def listar_arquivos_para_carregar(pasta):
    """LISTA OS CSV/TXT DA PASTA EXTRAIDA, APLICANDO O FILTRO DE PREFIXO."""
    pasta = Path(pasta)

    # PROCURA RECURSIVAMENTE POR ARQUIVOS .csv E .txt.
    arquivos = sorted(p for p in pasta.rglob("*") if p.suffix.lower() in (".csv", ".txt"))

    if not arquivos:
        raise RuntimeError(f"NENHUM ARQUIVO CSV/TXT ENCONTRADO EM {pasta}")

    # SE HOUVER FILTRO DE PREFIXO, MANTEM SO OS ARQUIVOS QUE COMECAM COM ELE.
    if PREFIXOS_ARQUIVOS_PROCESSADOS:
        prefixos = tuple(p.lower() for p in PREFIXOS_ARQUIVOS_PROCESSADOS)
        arquivos = [p for p in arquivos if p.name.lower().startswith(prefixos)]

        if not arquivos:
            raise RuntimeError(
                f"NENHUM ARQUIVO COM PREFIXO {PREFIXOS_ARQUIVOS_PROCESSADOS} ENCONTRADO EM {pasta}"
            )

    return arquivos


def conferir_colunas(caminho):
    """LE SO O CABECALHO DE UM CSV E CONFERE AS COLUNAS QUE OS NOTEBOOKS 03 E 04 PRECISAM.

    ESTA E A ETAPA DE CONFERENCIA MAIS IMPORTANTE DO NOTEBOOK. OS NOMES ESPERADOS
    ESTAO EM config.COLUNAS_OBRIGATORIAS_ACIDENTES E NUNCA FORAM VALIDADOS CONTRA
    O ARQUIVO REAL. SE ALGUM NOME MUDOU, DESCOBRIMOS AQUI.
    """
    encoding, separador = detectar_encoding_e_separador(caminho)

    # nrows=0 LE APENAS O CABECALHO, SEM CARREGAR NENHUMA LINHA DE DADO.
    cabecalho = pd.read_csv(caminho, sep=separador, encoding=encoding, nrows=0)
    colunas = list(cabecalho.columns)

    print(f"ARQUIVO   : {Path(caminho).name}")
    print(f"ENCODING  : {encoding}   SEPARADOR: {separador!r}")
    print(f"COLUNAS   : {len(colunas)}")
    print()

    # DESCOBRE QUAIS DAS COLUNAS ESPERADAS NAO APARECERAM.
    faltando = [c for c in config.COLUNAS_OBRIGATORIAS_ACIDENTES if c not in colunas]

    if not faltando:
        print("TODAS AS COLUNAS ESPERADAS FORAM ENCONTRADAS:")
        for coluna in config.COLUNAS_OBRIGATORIAS_ACIDENTES:
            print(f"   OK    {coluna}")
    else:
        print("ATENCAO: AS COLUNAS ABAIXO NAO EXISTEM NESTE ARQUIVO.")
        for coluna in faltando:
            print(f"   FALTA {coluna}")
        print()
        print("OS NOTEBOOKS 03 E 04 NAO VAO FUNCIONAR ATE QUE ISSO SEJA RESOLVIDO.")
        print("PROCURE O NOME EQUIVALENTE NA LISTA COMPLETA ABAIXO E CORRIJA AS")
        print("CONSTANTES COLUNA_* NO ARQUIVO config.py. E O UNICO LUGAR ONDE ELAS APARECEM.")

    print()
    print("LISTA COMPLETA DE COLUNAS DO ARQUIVO:")
    for i, coluna in enumerate(colunas, start=1):
        print(f"   {i:3}. {coluna}")

    return colunas, faltando


def carregar_csv_para_sqlite(caminho, conexao, tabela, prefixo=""):
    """LE UM CSV EM BLOCOS E GRAVA NA TABELA INDICADA, SUBSTITUINDO O CONTEUDO ANTERIOR."""
    encoding, separador = detectar_encoding_e_separador(caminho)
    total_linhas = 0
    primeiro_bloco = True

    # chunksize FAZ O PANDAS DEVOLVER UM LEITOR QUE ENTREGA O ARQUIVO EM PARTES.
    leitor = pd.read_csv(
        caminho,
        sep=separador,
        encoding=encoding,
        dtype=str,              # TUDO COMO TEXTO: EVITA O PANDAS ADIVINHAR TIPO ERRADO.
        chunksize=TAMANHO_CHUNK,
        on_bad_lines="skip",    # LINHA MALFORMADA E DESCARTADA EM VEZ DE PARAR TUDO.
        low_memory=False,
    )

    for bloco in leitor:
        # O PRIMEIRO BLOCO SUBSTITUI A TABELA; OS SEGUINTES SO ACRESCENTAM.
        bloco.to_sql(
            tabela,
            conexao,
            if_exists="replace" if primeiro_bloco else "append",
            index=False,
        )
        primeiro_bloco = False
        total_linhas += len(bloco)

        # MOSTRA O ANDAMENTO NA MESMA LINHA.
        print(f"\r{prefixo}{tabela}: {total_linhas} linhas...", end="")

    print()
    return total_linhas


print("FUNCOES AUXILIARES CARREGADAS.")

FUNCOES AUXILIARES CARREGADAS.


## 3. DESCOBRIR O ARQUIVO MAIS RECENTE

O PORTAL DE DADOS ABERTOS PUBLICA UMA NOVA VERSAO DOS DADOS DE TEMPOS EM TEMPOS.

ESTA ETAPA PERGUNTA AO PORTAL QUAIS ARQUIVOS EXISTEM, FICA COM OS QUE ESTAO ATIVOS E ESCOLHE O MAIS RECENTE PELA DATA.

A SAIDA MOSTRA A DATA DO RECURSO ESCOLHIDO. ANOTE ESSA DATA: ELA IDENTIFICA A VERSAO DOS DADOS QUE VOCE ESTA USANDO, E ISSO IMPORTA NA HORA DE EXPLICAR UM RESULTADO.

In [ ]:
# CONSULTA O PORTAL E ESCOLHE O RECURSO ATIVO MAIS RECENTE.
recurso = descobrir_recurso_mais_recente()

print(f"RECURSOS ATIVOS NO PACOTE : {recurso['total_ativos']}")
print(f"ESCOLHIDO                 : {recurso['nome']}")
print(f"DATA DO RECURSO           : {recurso['data']}")
print(f"LINK                      : {recurso['url']}")

RECURSOS ATIVOS NO PACOTE : 54
ESCOLHIDO                 : renaest_dabertos_20260712
DATA DO RECURSO           : 2026-07-20T12:24:09.968402
LINK                      : https://dados.transportes.gov.br/dataset/42e2320b-ea67-4fdc-896f-71363e043fc6/resource/84a8d44c-e9f2-40b2-88bc-089c036fd63e/download/renaest_dabertos_20260712.zip


## 4. BAIXAR O ARQUIVO ZIP

ESTA E A ETAPA DEMORADA DO NOTEBOOK. O ARQUIVO TEM CENTENAS DE MEGABYTES.

O DOWNLOAD E FEITO EM PEDACOS DE 10 MEGABYTES, PARA NAO CARREGAR TUDO NA MEMORIA DE UMA VEZ.

SE O ARQUIVO JA EXISTE NA PASTA TEMPORARIA E A FLAG `USAR_ARQUIVOS_LOCAIS_SE_EXISTIREM` ESTA LIGADA, O DOWNLOAD E PULADO.

In [ ]:
# CAMINHO ONDE O ZIP SERA GRAVADO.
caminho_zip = config.PASTA_TEMP / recurso["arquivo"]

# SE O ARQUIVO JA EXISTE E A FLAG PERMITE, REAPROVEITA E NAO BAIXA DE NOVO.
if USAR_ARQUIVOS_LOCAIS_SE_EXISTIREM and caminho_zip.exists():
    tamanho_mb = caminho_zip.stat().st_size / (1024 * 1024)
    print(f"ZIP JA EXISTE, DOWNLOAD PULADO: {caminho_zip}")
    print(f"TAMANHO: {tamanho_mb:.1f} MB")
else:
    print(f"BAIXANDO PARA: {caminho_zip}")
    baixar_arquivo(recurso["url"], caminho_zip)

BAIXANDO PARA: C:\Users\fabio\Documents\GitHub\dash-sinistros-renaest\projeto_v2\data\temp\renaest_dabertos_20260712.zip
PROGRESSO: 100.00%
DOWNLOAD CONCLUIDO.


## 5. EXTRAIR O ZIP

AQUI O CONTEUDO DO ZIP E COLOCADO EM UMA PASTA COM O MESMO NOME DO RECURSO.

ATENCAO A UM DETALHE QUE JA CAUSOU PROBLEMA NA VERSAO ANTERIOR DESTE CODIGO: O QUE SE ABRE COM `zipfile.ZipFile` E O **ARQUIVO .ZIP**, NAO A PASTA DE DESTINO. ABRIR A PASTA DE DESTINO FAZ O SCRIPT FALHAR OU EXTRAIR NADA.

In [ ]:
# PASTA ONDE O CONTEUDO DO ZIP SERA COLOCADO.
pasta_extraida = config.PASTA_TEMP / recurso["nome"]

# EXTRAI O CONTEUDO. NOTE QUE O PRIMEIRO ARGUMENTO E O CAMINHO DO .ZIP.
extrair_zip(caminho_zip, pasta_extraida)

# LISTA OS ARQUIVOS QUE SERAO CARREGADOS, DEPOIS DO FILTRO DE PREFIXO.
arquivos = listar_arquivos_para_carregar(pasta_extraida)

print()
print(f"ARQUIVOS QUE SERAO CARREGADOS ({len(arquivos)}):")
for caminho in arquivos:
    tamanho_mb = caminho.stat().st_size / (1024 * 1024)
    print(f"   {caminho.name}  ({tamanho_mb:.1f} MB)  ->  TABELA '{nome_tabela_limpo(caminho.stem)}'")

EXTRACAO CONCLUIDA EM: C:\Users\fabio\Documents\GitHub\dash-sinistros-renaest\projeto_v2\data\temp\renaest_dabertos_20260712

ARQUIVOS QUE SERAO CARREGADOS (1):
   Acidentes_DadosAbertos_20260712.csv  (2591.8 MB)  ->  TABELA 'acidentes'


## 6. CONFERIR AS COLUNAS ANTES DE CARREGAR

ESTA ETAPA EXISTE POR UM MOTIVO PRATICO.

OS NOTEBOOKS 03 E 04 PRECISAM DE CINCO COLUNAS ESPECIFICAS PARA FUNCIONAR:

- `num_acidente`: IDENTIFICADOR UNICO DE CADA ACIDENTE.
- `ano_acidente`: ANO, USADO PARA RECORTAR O QUE SERA PROCESSADO.
- `codigo_ibge`: CODIGO DO MUNICIPIO, TAMBEM USADO NO RECORTE.
- `end_acidente`: O ENDERECO EM TEXTO LIVRE, QUE SERA PADRONIZADO.
- `bairro_acidente`: O BAIRRO EM TEXTO LIVRE.

ESSES NOMES FORAM HERDADOS DA VERSAO EM API E **NUNCA FORAM CONFERIDOS CONTRA O ARQUIVO REAL**, PORQUE ATE ENTAO NENHUM SCRIPT DESTE PROJETO CHEGOU A CARREGAR O CSV DE ACIDENTES EM BANCO.

SE ALGUM NOME ESTIVER DIFERENTE, A CELULA ABAIXO MOSTRA QUAL FALTOU E LISTA TODAS AS COLUNAS DO ARQUIVO. AI BASTA CORRIGIR AS CONSTANTES `COLUNA_*` NO `config.py`, QUE E O UNICO LUGAR ONDE ESSES NOMES APARECEM.

In [ ]:
# CONFERE O CABECALHO DO PRIMEIRO ARQUIVO DA LISTA, SEM CARREGAR OS DADOS.
colunas_encontradas, colunas_faltando = conferir_colunas(arquivos[0])

ARQUIVO   : Acidentes_DadosAbertos_20260712.csv
ENCODING  : utf-8   SEPARADOR: ';'
COLUNAS   : 35

TODAS AS COLUNAS ESPERADAS FORAM ENCONTRADAS:
   OK    num_acidente
   OK    ano_acidente
   OK    codigo_ibge
   OK    end_acidente
   OK    bairro_acidente

LISTA COMPLETA DE COLUNAS DO ARQUIVO:
     1. num_acidente
     2. chv_localidade
     3. data_acidente
     4. uf_acidente
     5. ano_acidente
     6. mes_acidente
     7. mes_ano_acidente
     8. codigo_ibge
     9. dia_semana
    10. fase_dia
    11. tp_acidente
    12. cond_meteorologica
    13. end_acidente
    14. num_end_acidente
    15. cep_acidente
    16. bairro_acidente
    17. km_via_acidente
    18. latitude_acidente
    19. longitude_acidente
    20. hora_acidente
    21. tp_rodovia
    22. cond_pista
    23. tp_cruzamento
    24. tp_pavimento
    25. tp_curva
    26. lim_velocidade
    27. tp_pista
    28. ind_guardrail
    29. ind_cantcentral
    30. ind_acostamento
    31. qtde_acidente
    32. qtde_acid_com_obitos

## 7. CARREGAR OS CSV NO BANCO SQLITE

AGORA OS ARQUIVOS SAO LIDOS E GRAVADOS NO BANCO.

DOIS PONTOS IMPORTANTES:

### LEITURA EM BLOCOS

O CSV E LIDO EM BLOCOS DE 50 MIL LINHAS. ISSO EVITA QUE O PYTHON TENTE COLOCAR O ARQUIVO INTEIRO NA MEMORIA.

### TUDO E LIDO COMO TEXTO

USAMOS `dtype=str`, OU SEJA, TODA COLUNA E LIDA COMO TEXTO.

O MOTIVO E EVITAR QUE O PANDAS ADIVINHE O TIPO ERRADO. UM CODIGO DE MUNICIPIO COMO `5201405` SERIA LIDO COMO NUMERO E PODERIA PERDER UM ZERO A ESQUERDA EM OUTROS CASOS. NA INGESTAO, PRESERVAR O TEXTO ORIGINAL E MAIS SEGURO DO QUE INTERPRETAR.

### A TABELA E SUBSTITUIDA, NAO ACUMULADA

CADA EXECUCAO SUBSTITUI O CONTEUDO DA TABELA. RODAR ESTE NOTEBOOK DE NOVO NAO DUPLICA LINHA.

EM COMPENSACAO, SE VOCE JA RODOU O NOTEBOOK 03, A TABELA `acidentes_revisado` **NAO** E ATUALIZADA POR AQUI. ELA GUARDA A VERSAO QUE FOI REVISADA NA EPOCA.

In [ ]:
# ABRE A CONEXAO COM O BANCO DO PROJETO.
conexao = sqlite3.connect(str(config.BANCO))

# GUARDA QUANTAS LINHAS FORAM GRAVADAS EM CADA TABELA.
resumo_tabelas = {}

try:
    total_arquivos = len(arquivos)

    for indice, caminho in enumerate(arquivos, start=1):
        # DECIDE O NOME DA TABELA A PARTIR DO NOME DO ARQUIVO.
        tabela = nome_tabela_limpo(caminho.stem)

        reportar(
            etapa="carregando",
            progresso=(indice - 1) / total_arquivos * 100,
            mensagem=f"{caminho.name} -> tabela '{tabela}' ({indice}/{total_arquivos})",
        )

        # LE O ARQUIVO EM BLOCOS E GRAVA NA TABELA.
        linhas = carregar_csv_para_sqlite(
            caminho, conexao, tabela, prefixo=f"[{indice}/{total_arquivos}] "
        )

        resumo_tabelas[tabela] = linhas
finally:
    # FECHA A CONEXAO MESMO SE ALGO DER ERRADO NO MEIO.
    conexao.close()

print()
print("CARGA CONCLUIDA.")

[carregando]   0.0% Acidentes_DadosAbertos_20260712.csv -> tabela 'acidentes' (1/1)
[1/1] acidentes: 8637512 linhas...

CARGA CONCLUIDA.


## 8. RESUMO DA INGESTAO

ESTA ETAPA MOSTRA O QUE FICOU GRAVADO NO BANCO.

A SEGUNDA PARTE E A MAIS UTIL PARA O QUE VEM DEPOIS: ELA LISTA OS RECORTES DISPONIVEIS, OU SEJA, AS COMBINACOES DE ANO E MUNICIPIO QUE EXISTEM NOS DADOS.

OS NOTEBOOKS 03 E 04 PROCESSAM **UM MUNICIPIO POR EXECUCAO**, ENTAO E DAQUI QUE VOCE TIRA OS VALORES DE `ANO` E `CODIGO_IBGE` PARA COLOCAR NELES.

In [ ]:
# REABRE O BANCO SO PARA CONFERIR O RESULTADO.
conexao = sqlite3.connect(str(config.BANCO))

try:
    print("TABELAS GRAVADAS NESTA EXECUCAO:")
    for tabela, linhas in resumo_tabelas.items():
        print(f"   {tabela}: {linhas} linhas")

    print()
    print(f"BANCO: {config.BANCO}")
    print(f"TAMANHO: {config.BANCO.stat().st_size / (1024 * 1024):.1f} MB")

    # SE AS COLUNAS DE RECORTE EXISTEM, LISTA OS MUNICIPIOS COM MAIS ACIDENTES.
    if not colunas_faltando:
        print()
        print("RECORTES DISPONIVEIS (OS 15 MAIORES). USE ESTES VALORES NOS NOTEBOOKS 03 E 04:")
        print()
        consulta = (
            f"SELECT {config.COLUNA_ANO} AS ano, {config.COLUNA_MUNICIPIO} AS ibge, "
            f"count(*) AS total FROM acidentes "
            f"GROUP BY 1, 2 ORDER BY total DESC LIMIT 15"
        )
        recortes = pd.read_sql_query(consulta, conexao)
        print(recortes.to_string(index=False))
    else:
        print()
        print("O RESUMO POR RECORTE FOI PULADO PORQUE FALTAM COLUNAS. VEJA A ETAPA 6.")
finally:
    conexao.close()

TABELAS GRAVADAS NESTA EXECUCAO:
   acidentes: 8637512 linhas

BANCO: C:\Users\fabio\Documents\GitHub\dash-sinistros-renaest\projeto_v2\data\db\db_main.db
TAMANHO: 2782.3 MB

RECORTES DISPONIVEIS (OS 15 MAIORES). USE ESTES VALORES NOS NOTEBOOKS 03 E 04:

 ano    ibge  total
2025 3106200  88565
2024 3106200  84458
2019 3106200  82252
2023 3106200  78973
2018 3106200  74701
2022 3106200  69037
2021 3106200  62974
2020 3106200  60132
2023 5300108  57400
2022 5300108  55521
2025 5300108  53084
2024 5300108  47983
2021 5300108  47851
2023 3550308  46662
2024 3550308  45601


## PROXIMO PASSO

COM A TABELA `acidentes` CRIADA, O PROXIMO NOTEBOOK DA SEQUENCIA PARA OS ACIDENTES E O **03_revisao_enderecos.ipynb**.

MAS O **02_malha_viaria.ipynb** PODE SER RODADO A QUALQUER MOMENTO, INCLUSIVE ANTES DESTE: ELE NAO DEPENDE DOS DADOS DE ACIDENTE. O QUE ELE PRODUZ, A TABELA `vias_processadas`, SO E NECESSARIO NO NOTEBOOK 04.